In [1]:
import pandas as pd
import numpy as np
import talib
from datetime import datetime, timedelta, time, date

In [2]:
from api_call import *
api_options = ApiCall(data_type='Options')
api_spot = ApiCall(data_type='Equity')
api_expiry = ApiCall(data_type='ExpiryDates')

In [3]:
data = api_spot.get_data(["NIFTY_50"],start_date = '2012-01-01',till_today= True,resample_bool=True,resample_freq='15min',label = 'left')
data

,Datetime,Open,High,Low,Close,Volume,time,symbol
0,2012-01-02 09:15:00,4640.45,4642.80,4616.45,4639.85,0.0,09:15:00,NIFTY_50
1,2012-01-02 09:30:00,4639.55,4645.85,4633.55,4638.75,0.0,09:30:00,NIFTY_50
2,2012-01-02 09:45:00,4638.65,4640.90,4618.40,4619.80,0.0,09:45:00,NIFTY_50
3,2012-01-02 10:00:00,4619.85,4622.90,4610.00,4620.30,0.0,10:00:00,NIFTY_50
4,2012-01-02 10:15:00,4620.45,4621.15,4607.10,4617.90,0.0,10:15:00,NIFTY_50
...,...,...,...,...,...,...,...,...
83908,2025-08-05 14:15:00,24612.25,24622.00,24602.85,24621.25,6517542.0,14:15:00,NIFTY_50
83909,2025-08-05 14:30:00,24622.35,24631.85,24594.60,24623.45,8580736.0,14:30:00,NIFTY_50
83910,2025-08-05 14:45:00,24624.25,24629.90,24590.30,24611.10,9499517.0,14:45:00,NIFTY_50
83911,2025-08-05 15:00:00,24611.50,24671.15,24597.90,24666.65,22002710.0,15:00:00,NIFTY_50


In [4]:
def calculate_indicators(df):
    # TALib indicators
    df['ATR_20'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=20)
    df['ATR_25'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=25)
    df['EMA_135'] = talib.EMA(df['Close'], timeperiod=135)

    # Supertrend Calculation
    period = 10
    multiplier = 2.0
    high = df['High']
    low = df['Low']
    close = df['Close']
    atr = talib.ATR(high, low, close, timeperiod=period)
    hl2 = (high + low) / 2

    upper_basic = hl2 + multiplier * atr
    lower_basic = hl2 - multiplier * atr

    supertrend = [None] * len(df)
    direction = [1] * len(df)
    final_upper = [None] * len(df)
    final_lower = [None] * len(df)

    for i in range(len(df)):
        if i == 0:
            final_upper[i] = upper_basic[i]
            final_lower[i] = lower_basic[i]
            supertrend[i] = final_lower[i]
            continue

        if close[i-1] <= final_upper[i-1]:
            final_upper[i] = min(upper_basic[i], final_upper[i-1])
        else:
            final_upper[i] = upper_basic[i]

        if close[i-1] >= final_lower[i-1]:
            final_lower[i] = max(lower_basic[i], final_lower[i-1])
        else:
            final_lower[i] = lower_basic[i]
            
        # Assigning direction and supertrend value
        if close[i] > final_upper[i-1]:
            direction[i] = 1
            supertrend[i] = final_lower[i]
        elif close[i] < final_lower[i-1]:
            direction[i] = -1
            supertrend[i] = final_upper[i]
        else:
            direction[i] = direction[i-1]
            supertrend[i] = final_lower[i] if direction[i] == 1 else final_upper[i]

    # Add results to dataframe
    df['Supertrend'] = pd.Series(supertrend, index=df.index)
    df['Supertrend_trend'] = pd.Series(['up' if d == 1 else 'down' for d in direction], index=df.index)

    return df


In [5]:
nf=data.copy()
nf=calculate_indicators(nf)
nf

,Datetime,Open,High,Low,Close,Volume,time,symbol,ATR_20,ATR_25,EMA_135,Supertrend,Supertrend_trend
0,2012-01-02 09:15:00,4640.45,4642.80,4616.45,4639.85,0.0,09:15:00,NIFTY_50,NaN,NaN,NaN,NaN,up
1,2012-01-02 09:30:00,4639.55,4645.85,4633.55,4638.75,0.0,09:30:00,NIFTY_50,NaN,NaN,NaN,NaN,up
2,2012-01-02 09:45:00,4638.65,4640.90,4618.40,4619.80,0.0,09:45:00,NIFTY_50,NaN,NaN,NaN,NaN,up
3,2012-01-02 10:00:00,4619.85,4622.90,4610.00,4620.30,0.0,10:00:00,NIFTY_50,NaN,NaN,NaN,NaN,up
4,2012-01-02 10:15:00,4620.45,4621.15,4607.10,4617.90,0.0,10:15:00,NIFTY_50,NaN,NaN,NaN,NaN,up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
83908,2025-08-05 14:15:00,24612.25,24622.00,24602.85,24621.25,6517542.0,14:15:00,NIFTY_50,33.112984,34.498942,24724.269299,24667.113747,down
83909,2025-08-05 14:30:00,24622.35,24631.85,24594.60,24623.45,8580736.0,14:30:00,NIFTY_50,33.319835,34.608984,24722.786662,24667.113747,down
83910,2025-08-05 14:45:00,24624.25,24629.90,24590.30,24611.10,9499517.0,14:45:00,NIFTY_50,33.633843,34.808625,24721.144211,24667.113747,down
83911,2025-08-05 15:00:00,24611.50,24671.15,24597.90,24666.65,22002710.0,15:00:00,NIFTY_50,35.614651,36.346280,24720.342826,24667.113747,down


In [6]:
nf=nf.dropna()
nf=nf.reset_index(drop=True)
nf.head(50)

,Datetime,Open,High,Low,Close,Volume,time,symbol,ATR_20,ATR_25,EMA_135,Supertrend,Supertrend_trend
0,2012-01-07 12:15:00,4752.35,4752.55,4747.55,4747.70,0.0,12:15:00,NIFTY_50,11.961292,12.265629,4719.027037,4730.754709,up
1,2012-01-07 12:30:00,4747.70,4749.00,4743.35,4744.20,0.0,12:30:00,NIFTY_50,11.645727,12.001004,4719.397228,4730.754709,up
2,2012-01-07 12:45:00,4743.60,4746.90,4743.60,4746.90,0.0,12:45:00,NIFTY_50,11.228441,11.652964,4719.801680,4730.754709,up
3,2012-01-09 09:15:00,4743.90,4743.90,4704.25,4704.25,0.0,09:15:00,NIFTY_50,12.799519,12.892845,4719.572979,4748.972358,down
4,2012-01-09 09:30:00,4704.35,4708.45,4695.80,4698.15,0.0,09:30:00,NIFTY_50,12.792043,12.883132,4719.257935,4727.062622,down
5,2012-01-09 09:45:00,4698.70,4711.75,4698.65,4711.25,0.0,09:45:00,NIFTY_50,12.832441,12.911806,4719.140171,4727.062622,down
6,2012-01-09 10:00:00,4711.25,4711.65,4701.80,4706.05,0.0,10:00:00,NIFTY_50,12.683319,12.789334,4718.947669,4727.062622,down
7,2012-01-09 10:15:00,4705.90,4711.45,4701.70,4709.90,0.0,10:15:00,NIFTY_50,12.536653,12.667761,4718.814615,4727.062622,down
8,2012-01-09 10:30:00,4710.00,4715.50,4707.60,4709.45,0.0,10:30:00,NIFTY_50,12.304820,12.477050,4718.676900,4727.062622,down
9,2012-01-09 10:45:00,4709.20,4712.25,4703.90,4707.50,0.0,10:45:00,NIFTY_50,12.107079,12.311968,4718.512534,4727.062622,down


In [7]:
#First Preparing a new dataframe with datetime indexing
new_df=nf.copy()
new_df['Datetime']=pd.to_datetime(new_df['Datetime'])
new_df.set_index('Datetime', inplace=True)
new_df.sort_index(inplace=True)

#Function to compute previous date
unique_dates = sorted({ts.date() for ts in new_df.index})

def prev_trading_day(curr_date):
    prior = [d for d in unique_dates if d<curr_date]
    return prior[-1] if prior else None

#Initializing variables
trades=[]
tp=sl=None
position=None
initial_atr=None
entry_dt=exit_dt=None
entry_px=exit_px=None
last_tp_hit=None

for idx,row in new_df.iterrows():
    curr_time=idx.time()
    curr_date=idx.date()

    
    if curr_time == time(9,15) :
        upper_range=row['High']
        lower_range=row['Low']
        prev_day=prev_trading_day(curr_date)
        if prev_day :
            #print(new_df.loc[prev_day.strftime('%Y-%m-%d'), ['Close', 'ATR_25']])
            atr_series=new_df.loc[prev_day.strftime('%Y-%m-%d'),'ATR_25']
            if not atr_series.empty:
                initial_atr=atr_series.iloc[-1]


    #For pre-existing Long Position
    if position == "Long" :
        if row['Close']>=tp:
            atr_new=row['ATR_20']
            sl_dist=atr_new*2.25
            sl=row['Close']-sl_dist
            tp=row['Close']+1.5*sl_dist
            last_tp_hit=row['Close']
        elif row['Close']<=sl and row['Supertrend_trend']=="down" and row['Close']<=row['EMA_135']:
            exit_px=row['Close']
            exit_dt=idx
            trades.append({
                'EDT' : entry_dt,
                'Exit_Datetime' : exit_dt,
                'Entry_Value' : entry_px,
                'Exit_Value' : exit_px,
                'PnL' : exit_px-entry_px,
                'Position' : 'Long',
                'Reason' : "Stoploss Hit"
            })
            position=None
        elif row['Close']<=sl and (last_tp_hit - row['Close']) / last_tp_hit >= 0.02:
            exit_px=row['Close']
            exit_dt=idx
            trades.append({
                'EDT' : entry_dt,
                'Exit_Datetime' : exit_dt,
                'Entry_Value' : entry_px,
                'Exit_Value' : exit_px,
                'PnL' : exit_px-entry_px,
                'Position' : 'Long',
                'Reason' : "2% Reversal"
            })
            position=None
    # For Pre-existing Short Position
    if position == "Short" : 
        if row['Close']<=tp : 
            atr_new=row['ATR_20']
            sl_dist=atr_new*2.25
            sl=row['Close']+sl_dist
            tp=row['Close']-1.5*sl_dist
            last_tp_hit=row['Close']
        elif row['Close']>=sl and row['Supertrend_trend']=="up" and row['Close']>=row['EMA_135'] :
            exit_px=row['Close']
            exit_dt=idx
            trades.append({
                'EDT' : entry_dt,
                'Exit_Datetime' : exit_dt,
                'Entry_Value' : entry_px,
                'Exit_Value' : exit_px,
                'PnL' : entry_px-exit_px,
                'Position' : 'Short',
                'Reason' : 'Stoploss Hit'
            })
            position=None
        elif row['Close']>=sl and (row['Close']-last_tp_hit) / last_tp_hit >= 0.02:
            exit_px=row['Close']
            exit_dt=idx
            trades.append({
                'EDT' : entry_dt,
                'Exit_Datetime' : exit_dt,
                'Entry_Value' : entry_px,
                'Exit_Value' : exit_px,
                'PnL' : entry_px-exit_px,
                'Position' : 'Short',
                'Reason' : '2% Reversal'
            })
            position=None

    #If there is no position, then we might consider taking a new position
    if position is None and initial_atr is not None and time(9,30) <= curr_time : #<= time(15,15)''' :
        if row['Close']>upper_range and row['Supertrend_trend'] == "up" :
            position="Long"
        elif row['Close']<lower_range and row['Supertrend_trend'] == 'down' :
            position="Short"
        if position : 
            entry_dt=idx
            sl_dist=initial_atr*2.25
            entry_px=row['Close']
            last_tp_hit=entry_px
            if position == "Long" : 
                sl=entry_px-sl_dist
                tp=entry_px+1.5*sl_dist
            else : 
                sl=entry_px+sl_dist
                tp=entry_px-1.5*sl_dist
    

trades_df=pd.DataFrame(trades)
trades_df



,EDT,Exit_Datetime,Entry_Value,Exit_Value,PnL,Position,Reason
0,2012-01-09 09:30:00,2012-01-09 12:30:00,4698.15,4732.10,-33.95,Short,Stoploss Hit
1,2012-01-09 13:15:00,2012-01-30 14:15:00,4748.05,5107.80,359.75,Long,Stoploss Hit
2,2012-01-30 14:15:00,2012-01-31 09:15:00,5107.80,5138.65,-30.85,Short,Stoploss Hit
3,2012-01-31 09:45:00,2012-02-22 14:15:00,5153.25,5526.10,372.85,Long,Stoploss Hit
4,2012-02-22 14:15:00,2012-02-23 14:15:00,5526.10,5525.65,0.45,Short,Stoploss Hit
...,...,...,...,...,...,...,...
1173,2025-07-22 11:00:00,2025-07-23 12:00:00,25072.00,25151.85,-79.85,Short,Stoploss Hit
1174,2025-07-23 12:00:00,2025-07-24 11:45:00,25151.85,25072.30,-79.55,Long,Stoploss Hit
1175,2025-07-24 11:45:00,2025-07-30 11:15:00,25072.30,24898.70,173.60,Short,Stoploss Hit
1176,2025-07-30 11:15:00,2025-07-31 09:15:00,24898.70,24709.40,-189.30,Long,Stoploss Hit


In [8]:
print(trades_df['PnL'].sum())
print(trades_df['PnL'].mean())

39987.90000000004
33.94558573853993


In [9]:
# Ensure EDT is datetime
sample=trades_df.copy()
sample['EDT'] = pd.to_datetime(sample['EDT'])

# Extract year and create a new column
sample['Year'] = sample['EDT'].dt.year

# Group by year and sum PnL
pnl_by_year = sample.groupby('Year')['PnL'].sum().reset_index()

# Optional: Rename column for clarity
pnl_by_year.rename(columns={'PnL': 'Total_PnL'}, inplace=True)

pnl_by_year


,Year,Total_PnL
0,2012,796.40
1,2013,2311.00
2,2014,1976.30
3,2015,2650.65
4,2016,1169.15
5,2017,910.45
6,2018,2789.00
7,2019,1802.90
8,2020,7346.30
9,2021,4088.60


In [10]:
trades_df['Position'].value_counts()

Position
Short    615
Long     563
Name: count, dtype: int64

In [11]:
trades_df['Reason'].value_counts()

Reason
Stoploss Hit    1144
2% Reversal       34
Name: count, dtype: int64

In [12]:
expiry_dates = api_expiry.get_data(symbol = "NIFTY",segment = 'O') # Segment = 'F' for Futures and 'O' for Options
expiry_dates = expiry_dates.loc[expiry_dates["Date"] >= pd.to_datetime('2012-01-01')].reset_index(drop = True)
expiry_dates

,Date
0,2012-01-25
1,2012-02-23
2,2012-03-29
3,2012-04-26
4,2012-05-31
...,...
443,2028-06-29
444,2028-12-26
445,2028-12-28
446,2029-06-28


In [13]:
trades_copy=trades_df.copy()
exp_dates=expiry_dates[expiry_dates["Date"]<=pd.to_datetime("2025-09-02")]
exp_dates['YearMonth']=exp_dates['Date'].dt.to_period('M')
month_end_exp=exp_dates.groupby('YearMonth')['Date'].max().reset_index()
month_end_exp=month_end_exp.sort_values('Date').reset_index(drop = True)

trades_copy['Index_Date']=trades_copy['EDT'].dt.date

trades_copy = pd.merge_asof(
    trades_copy,
    month_end_exp.rename(columns={'Date': 'expiry_date'}),
    left_on='EDT',
    right_on='expiry_date',
    direction='forward'
)
trades_copy.drop(columns=["YearMonth"], inplace=True)

C:\Users\Sanidhya.Raj\AppData\Local\Temp\ipykernel_17236\1560256371.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exp_dates['YearMonth']=exp_dates['Date'].dt.to_period('M')


In [14]:
month_end_exp

,YearMonth,Date
0,2012-01,2012-01-25
1,2012-02,2012-02-23
2,2012-03,2012-03-29
3,2012-04,2012-04-26
4,2012-05,2012-05-31
...,...,...
160,2025-05,2025-05-29
161,2025-06,2025-06-26
162,2025-07,2025-07-31
163,2025-08,2025-08-28


In [15]:
trades_copy.head(10)

,EDT,Exit_Datetime,Entry_Value,Exit_Value,PnL,Position,Reason,Index_Date,expiry_date
0,2012-01-09 09:30:00,2012-01-09 12:30:00,4698.15,4732.10,-33.95,Short,Stoploss Hit,2012-01-09,2012-01-25
1,2012-01-09 13:15:00,2012-01-30 14:15:00,4748.05,5107.80,359.75,Long,Stoploss Hit,2012-01-09,2012-01-25
2,2012-01-30 14:15:00,2012-01-31 09:15:00,5107.80,5138.65,-30.85,Short,Stoploss Hit,2012-01-30,2012-02-23
3,2012-01-31 09:45:00,2012-02-22 14:15:00,5153.25,5526.10,372.85,Long,Stoploss Hit,2012-01-31,2012-02-23
4,2012-02-22 14:15:00,2012-02-23 14:15:00,5526.10,5525.65,0.45,Short,Stoploss Hit,2012-02-22,2012-02-23
5,2012-02-23 14:15:00,2012-02-23 15:00:00,5525.65,5486.85,-38.80,Long,Stoploss Hit,2012-02-23,2012-03-29
6,2012-02-23 15:00:00,2012-02-29 09:15:00,5486.85,5443.30,43.55,Short,Stoploss Hit,2012-02-23,2012-03-29
7,2012-02-29 10:45:00,2012-03-02 12:15:00,5415.40,5384.70,30.70,Short,Stoploss Hit,2012-02-29,2012-03-29
8,2012-03-02 12:15:00,2012-03-02 13:15:00,5384.70,5342.90,-41.80,Long,Stoploss Hit,2012-03-02,2012-03-29
9,2012-03-05 09:45:00,2012-03-06 10:45:00,5298.00,5368.55,-70.55,Short,Stoploss Hit,2012-03-05,2012-03-29


In [16]:
from tqdm import tqdm

def next_month_end_exp(curr_exp):
    next_exp = month_end_exp[month_end_exp['Date'] > curr_exp]['Date']
    return next_exp.iloc[0] if not next_exp.empty else None

def resample_option_df(df):
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df.set_index('Datetime', inplace=True)
    df = df.resample('15min').last()
    return df

def get_options_df(start_date,strike_price,option_type, expiry) : 
    ticker='NIFTY'
    df=api_options.get_data(
            symbol=[ticker],
            start_date=start_date.date(),
            end_date=start_date.date(),
            start_expiry=expiry,
            strike_price=[strike_price],
            option_type=option_type
        )
    return df

def generate_options_trades(trades_copy) : 
    option_trades=[]
    skipped_rows=[]


    for index,trade in tqdm(trades_copy.iterrows(), total=len(trades_copy), desc="Processing Trades") : 
        edt=trade['EDT']
        exit_dt=trade['Exit_Datetime']
        entry_val=trade['Entry_Value']
        position=trade['Position']
        strike_price=round(entry_val/100)*100
        itm_ce=entry_val-100
        itm_pe=entry_val
        itm_ce=round(itm_ce/100)*100
        itm_pe=round(itm_pe/100)*100
        curr_exp=trade['expiry_date']
        #check=pd.Timestamp(f"{curr_exp.date()} 15:15:00")

        segments=[]
        while curr_exp.date()<exit_dt.date() : 
            new_ext=pd.Timestamp(f"{curr_exp.date()} 15:15:00")
            segments.append((edt,new_ext,curr_exp))
            edt=new_ext
            curr_exp=next_month_end_exp(curr_exp)
                
        segments.append((edt,exit_dt,curr_exp))
        
        
        for start,end,exp in segments : 
            try : 
                try : 
                    ev_ce_df= get_options_df(start, itm_ce, "CE", exp)
                    if ev_ce_df is None or len(ev_ce_df) == 0:
                            raise ValueError("No CE data returned")
                except Exception as ce_err:
                    raise Exception(f"CE_EV fetch error: {ce_err}")
                    
                try : 
                    ex_ce_df=get_options_df(end,itm_ce, "CE", exp)
                    if ex_ce_df is None or len(ex_ce_df) == 0:
                            raise ValueError("No CE data returned")
                except Exception as ce_err:
                    raise Exception(f"CE_EX fetch error: {ce_err}")
                    
                try : 
                    ev_pe_df=get_options_df(start, itm_pe, "PE", exp)
                    if ev_pe_df is None or len(ev_pe_df) == 0:
                            raise ValueError("No PE data returned")
                except Exception as pe_err:
                    raise Exception(f"PE_EV fetch error: {pe_err}")
                    
                try : 
                    ex_pe_df=get_options_df(end,itm_pe,"PE",exp)
                    if ex_pe_df is None or len(ex_pe_df) == 0:
                            raise ValueError("No PE data returned")
                except Exception as pe_err:
                    raise Exception(f"PE_EX fetch error: {pe_err}")
                    

                ev_ce_15 = resample_option_df(ev_ce_df)
                ex_ce_15 = resample_option_df(ex_ce_df)
                ev_pe_15 = resample_option_df(ev_pe_df)
                ex_pe_15 = resample_option_df(ex_pe_df)


                ev_ce = ev_ce_15.at[start, 'Close'] if start in ev_ce_15.index else None
                ex_ce = ex_ce_15.at[end, 'Close'] if end in ex_ce_15.index else None
                ev_pe = ev_pe_15.at[start, 'Close'] if start in ev_pe_15.index else None
                ex_pe = ex_pe_15.at[end, 'Close'] if end in ex_pe_15.index else None

                missing_reasons = []
                if ev_ce is None or pd.isna(ev_ce) :
                    missing_reasons.append(f"EV_CE not found at {start}")
                if ex_ce is None or pd.isna(ex_ce):
                    missing_reasons.append(f"EX_CE not found at {end}")
                if ev_pe is None or pd.isna(ev_pe):
                    missing_reasons.append(f"EV_PE not found at {start}")
                if ex_pe is None or pd.isna(ev_pe):
                    missing_reasons.append(f"EX_PE not found at {end}")

                if not missing_reasons:
                    if position == 'Long':
                        ce_pl = ex_ce - ev_ce
                        pe_pl = ev_pe - ex_pe
                    elif position == 'Short':
                        ce_pl = ev_ce - ex_ce
                        ex_pe=ev_pe=0
                        pe_pl = 0

                    total_pl = ce_pl + pe_pl
                    option_trades.append({
                        'EDT': start,
                        'Exit_Datetime': end,
                        'CE_Strike_Price': itm_ce,
                        'PE_Strike_Price' : itm_pe,
                        'Expiry_Date': exp,
                        'EV_CE': ev_ce,
                        'EV_PE': ev_pe,
                        'EX_CE': ex_ce,
                        'EX_PE': ex_pe,
                        'CE_PL': ce_pl,
                        'PE_PL': pe_pl,
                        'Total_PL': total_pl,
                        'Position': position
                    })
                else:
                    skipped_rows.append({
                        'EDT': start,
                        'Exit_Datetime': end,
                        'CE_Strike_Price': itm_ce,
                        'PE_Strike_Price' : itm_pe,
                        'Expiry_Date': exp,
                        'Position': position,
                        'Reason': "; ".join(missing_reasons)
                    })  
                

            except Exception as e:
                skipped_rows.append({
                    'EDT': start,
                    'Exit_Datetime': end,
                    'CE_Strike_Price': itm_ce,
                    'PE_Strike_Price' : itm_pe,
                    'Expiry_Date': exp,
                    'Position': position,
                    'Reason': str(e)
                })


    return pd.DataFrame(option_trades), pd.DataFrame(skipped_rows)





In [17]:
o,s=generate_options_trades(trades_copy)
o

Processing Trades: 100%|██████████| 1178/1178 [03:14<00:00,  6.06it/s]


,EDT,Exit_Datetime,CE_Strike_Price,PE_Strike_Price,Expiry_Date,EV_CE,EV_PE,EX_CE,EX_PE,CE_PL,PE_PL,Total_PL,Position
0,2012-01-09 09:30:00,2012-01-09 12:30:00,4600,4700,2012-01-25,168.00,0.00,192.00,0.00,-24.00,0.00,-24.00,Short
1,2012-01-09 13:15:00,2012-01-25 15:15:00,4600,4700,2012-01-25,206.15,76.95,555.00,0.05,348.85,76.90,425.75,Long
2,2012-01-25 15:15:00,2012-01-30 14:15:00,4600,4700,2012-02-23,573.75,13.35,531.25,12.95,-42.50,0.40,-42.10,Long
3,2012-01-30 14:15:00,2012-01-31 09:15:00,5000,5100,2012-02-23,189.00,0.00,209.60,0.00,-20.60,0.00,-20.60,Short
4,2012-01-31 09:45:00,2012-02-22 14:15:00,5100,5200,2012-02-23,150.30,124.40,440.00,0.55,289.70,123.85,413.55,Long
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1269,2025-07-22 11:00:00,2025-07-23 12:00:00,25000,25100,2025-07-31,212.70,0.00,255.85,0.00,-43.15,0.00,-43.15,Short
1270,2025-07-23 12:00:00,2025-07-24 11:45:00,25100,25200,2025-07-31,187.35,138.00,133.50,186.55,-53.85,-48.55,-102.40,Long
1271,2025-07-24 11:45:00,2025-07-30 11:15:00,25000,25100,2025-07-31,191.60,0.00,37.70,0.00,153.90,0.00,153.90,Short
1272,2025-07-30 11:15:00,2025-07-31 09:15:00,24800,24900,2025-07-31,138.75,65.40,21.05,210.85,-117.70,-145.45,-263.15,Long


In [18]:
s

,EDT,Exit_Datetime,CE_Strike_Price,PE_Strike_Price,Expiry_Date,Position,Reason
0,2014-12-18 14:45:00,2014-12-24 11:45:00,8100,8200,2014-12-24,Long,CE_EX fetch error: No records found in the dat...
1,2015-09-08 14:15:00,2015-09-10 09:15:00,7600,7700,2015-09-24,Long,CE_EX fetch error: No records found in the dat...
2,2017-05-05 11:45:00,2017-05-08 09:45:00,9200,9300,2017-05-25,Short,PE_EV fetch error: No records found in the dat...
3,2017-05-18 14:00:00,2017-05-19 09:15:00,9400,9500,2017-05-25,Short,CE_EX fetch error: No records found in the dat...
4,2017-09-21 09:45:00,2017-09-21 14:15:00,10000,10100,2017-09-28,Short,CE_EX fetch error: No records found in the dat...
5,2018-07-17 14:00:00,2018-07-18 13:45:00,10900,11000,2018-07-26,Long,PE_EX fetch error: No records found in the dat...
6,2018-07-18 13:45:00,2018-07-20 12:00:00,10900,11000,2018-07-26,Short,PE_EV fetch error: No records found in the dat...
7,2019-01-31 10:30:00,2019-02-08 12:45:00,10600,10700,2019-02-28,Long,CE_EV fetch error: No records found in the dat...
8,2019-04-03 15:00:00,2019-04-09 13:45:00,11500,11600,2019-04-25,Short,PE_EX fetch error: No records found in the dat...
9,2019-08-08 14:15:00,2019-08-13 13:45:00,10900,11000,2019-08-29,Long,CE_EV fetch error: No records found in the dat...


In [19]:
def average_bars_held(kpi_df):
    bars_held_list = []

    for _, row in tqdm(kpi_df.iterrows(), total=len(kpi_df), desc="Calculating Bars Held"):
        start_dt = pd.to_datetime(row['EDT'])
        end_dt = pd.to_datetime(row['Exit_Datetime'])
        data['Datetime'] = pd.to_datetime(data['Datetime'])
        
        bars_held = data[(data['Datetime'] >= start_dt) & (data['Datetime'] <= end_dt)].shape[0]

        bars_held_list.append(bars_held)

    kpi_df["No. of Bars Held"] = bars_held_list
    return kpi_df

def calculate_roi(kpi_df) :
    kpi_df['ROI']= 75*kpi_df['Total_PL']/450000
    #kpi_df['ROI']=kpi_df['ROI'].apply(lambda x: f"{x:.2f}%")
    return kpi_df

def max_consecutive_profits(kpi_df) :
    profit=loss=0
    max_profit=max_loss=0
    for _, row in tqdm(kpi_df.iterrows(), total=len(kpi_df), desc="Calculating Max_Consecutive_Profit") :
        PL=row['Total_PL']
        if PL>0 : 
            profit+=1
            max_loss=max(max_loss,loss)
            loss=0
        else :
            loss+=1 
            max_profit=max(max_profit,profit)
            profit=0
    return max_profit,max_loss

def calculate_Drawdown(kpi_df) : 
    kpi_df['1']=1+kpi_df["Total_PL"].cumsum()
    kpi_df['0.0001']=0.0001+kpi_df['ROI'].cumsum()
    dd_list=[]
    dd_roi_list=[]
    current_peak=1
    current_roi_peak=0.0001
    for value in kpi_df['1'] :
        if value>=current_peak:
            current_peak=value
            dd_list.append(0)
        else :
            dd_list.append(value-current_peak)

    for value in kpi_df['0.0001'] :
        if value>=current_roi_peak:
            current_roi_peak=value
            dd_roi_list.append(0)
        else :
            dd_roi_list.append(value-current_roi_peak)

    kpi_df["Drawdown"]=dd_list
    kpi_df["DD_ROI"]= dd_roi_list
    kpi_df['Cumulative_Max'] = kpi_df['1'].cummax()
    kpi_df['Drawdown_Percentage']=(kpi_df["Drawdown"]/kpi_df['Cumulative_Max'])*100
    kpi_df.drop(columns=['1','0.0001'], inplace=True)
    return kpi_df

def calculate_cagr(kpi_df):
    begin_value=1
    end_value=kpi_df['Total_PL'].sum()
    first_date = kpi_df['EDT'].min().date()
    last_date = kpi_df['Exit_Datetime'].max().date()
    days       = (last_date - first_date).days
    years      = days / 365.25
    cagr = (end_value / begin_value) ** (1 / years) - 1
    return cagr

In [20]:

def evaluate_kpi(kpi_df) : 

    Total_Trades= len(kpi_df)
    Total_PL=kpi_df['Total_PL'].sum()
    Average_PL_per_trade= kpi_df['Total_PL'].mean()

    kpi_df=average_bars_held(kpi_df)
    Avg_bars_held=kpi_df['No. of Bars Held'].mean()

    kpi_df=calculate_roi(kpi_df)
    ROI_Profitable_Trades=kpi_df[kpi_df['ROI']>0]['ROI'].sum()
    ROI_Loss_Trades=kpi_df[kpi_df['ROI']<0]['ROI'].sum()
    Overall_ROI=kpi_df['ROI'].sum()

    #Avg_Profit_by_Loss_Percentage= kpi_df[kpi_df['ROI']>0]['ROI'].mean()/abs(kpi_df[kpi_df['ROI']<0]['ROI']).mean()
    Avg_Profit_by_Loss_Percentage=kpi_df['ROI'].mean()*100
    Winners=kpi_df[kpi_df['Total_PL']>0].shape[0]/Total_Trades
    Total_Profit= kpi_df[kpi_df['Total_PL']>0]['Total_PL'].sum()
    Avg_Profit= kpi_df[kpi_df['Total_PL']>0]['Total_PL'].mean()
    Avg_Profit_percentage = kpi_df[kpi_df['ROI']>0]['ROI'].mean()

    Avg_bars_held_profitable_trades = kpi_df[kpi_df['Total_PL']>0]['No. of Bars Held'].mean()

    Maximum_Consecutive_Profit_Days,Maximum_Consecutive_Loss_Days=max_consecutive_profits(kpi_df)

    Largest_Win= kpi_df['Total_PL'].max()
    Largest_Win_Percentage = float(kpi_df.loc[kpi_df['Total_PL'].idxmax(), 'ROI'] * 100)
    Bars_in_Largest_Winning_Trade = kpi_df[kpi_df['Total_PL']==kpi_df['Total_PL'].max()]['No. of Bars Held']

    Losers = kpi_df[kpi_df['Total_PL']<0].shape[0]/Total_Trades
    Total_Loss= kpi_df[kpi_df['Total_PL']<0]['Total_PL'].sum()
    Avg_Loss=kpi_df[kpi_df['Total_PL']<0]['Total_PL'].mean()
    Avg_Loss_Percentage = kpi_df[kpi_df['ROI']<0]['ROI'].mean()

    Avg_bars_held_Losing_trades = kpi_df[kpi_df['Total_PL']<0]['No. of Bars Held'].mean()

    Largest_Loss= kpi_df['Total_PL'].min()
    Bars_in_Largest_Losing_Trade = kpi_df[kpi_df['Total_PL']==kpi_df['Total_PL'].min()]['No. of Bars Held']

    kpi_df=calculate_Drawdown(kpi_df)
    Max_Drawdown=kpi_df['Drawdown'].min()
    Max_Drawdown_Percentage = kpi_df['Drawdown_Percentage'].min()
    Max_Drawdown_ROI = kpi_df['DD_ROI'].min()

    Recovery_Factor= abs(kpi_df['Total_PL'].sum())/abs(Max_Drawdown) if Max_Drawdown != 0 else 0

    cagr=calculate_cagr(kpi_df)*100
    CAR_MDD_Ratio= cagr/abs(Max_Drawdown_Percentage)

    Profit_Factor= kpi_df[kpi_df['Total_PL']>0]['Total_PL'].sum()/abs(kpi_df[kpi_df['Total_PL']<0]['Total_PL'].sum())
    Payoff_Ratio=abs(Avg_Profit_percentage/Avg_Loss_Percentage)
    Risk_Reward_Ratio = f"1:{abs(Avg_Profit_percentage / Avg_Loss_Percentage):.2f}" if Avg_Loss_Percentage != 0 else "1:0"
    returns=kpi_df['ROI']*100
    sharpe_ratio = ((returns.mean()*252-7) / (returns.std()* np.sqrt(252)))  if returns.std() != 0 else 0

    print(f"\n===== Strategy KPIs =====")
    print(f"Total Trades                  : {Total_Trades}")
    print(f"Total P&L                     : {Total_PL:.2f}")
    print(f"Average P&L per Trade         : {Average_PL_per_trade:.2f}")
    print(f"Average Bars Held             : {Avg_bars_held:.2f}")
    print(f"Average Bars Held (Profits)   : {Avg_bars_held_profitable_trades:.2f}")
    print(f"Average Bars Held (Losses)    : {Avg_bars_held_Losing_trades:.2f}")
    print(f"Overall ROI                   : {Overall_ROI*100:.2f}%")
    #print(f"ROI on Profitable Trades      : {ROI_Profitable_Trades*100:.2f}%")
    #print(f"ROI on Losing Trades          : {ROI_Loss_Trades*100:.2f}%")
    print(f"Average Profit %              : {Avg_Profit_percentage*100:.2f}%")
    print(f"Average Loss %                : {Avg_Loss_Percentage*100:.2f}%")
    print(f"Payoff Ratio (P/L%)           : {Payoff_Ratio:.2f}")
    print(f"Risk-Reward Ratio (L/P%)      : {Risk_Reward_Ratio}")
    print(f"Avg Profit/Loss %             : {Avg_Profit_by_Loss_Percentage:.2f}%")
    print(f"Winners %                     : {Winners*100:.2f}%")
    print(f"Losers %                      : {Losers*100:.2f}%")
    print(f"Total Profit                  : {Total_Profit:.2f}")
    print(f"Total Loss                    : {Total_Loss:.2f}")
    print(f"Average Profit                : {Avg_Profit:.2f}")
    print(f"Average Loss                  : {Avg_Loss:.2f}")
    print(f"Largest Win                   : {Largest_Win:.2f}")
    print(f"Largest Win Percentage        : {Largest_Win_Percentage:.2f}%")
    print(f"Largest Loss                  : {Largest_Loss:.2f}")
    print(f"Bars in Largest Win Trade     : {Bars_in_Largest_Winning_Trade.values[0]}")
    print(f"Bars in Largest Loss Trade    : {Bars_in_Largest_Losing_Trade.values[0]}")
    print(f"Max Drawdown                  : {Max_Drawdown:.2f}")
    print(f"Max Drawdown %                : {Max_Drawdown_Percentage}")
    print(f"Max Drawdown ROI              : {Max_Drawdown_ROI:.2f}")
    print(f"Recovery Factor               : {Recovery_Factor:.2f}")
    print(f"CAGR                          : {cagr:.2f}%")
    print(f"CAR/MDD Ratio                 : {CAR_MDD_Ratio:.2f}")
    print(f"Profit Factor                 : {Profit_Factor:.2f}")
    print(f"Max Consecutive Profit_Trades : {Maximum_Consecutive_Profit_Days}")
    print(f"Max Consecutive Loss_Trades   : {Maximum_Consecutive_Loss_Days}")
    print(f"Sharpe Ratio                  : {sharpe_ratio}")


In [21]:
kpi_df=o.copy()
evaluate_kpi(kpi_df)

Calculating Max_Consecutive_Profit: 100%|██████████| 1274/1274 [00:00<00:00, 14305.64it/s]



===== Strategy KPIs =====
Total Trades                  : 1274
Total P&L                     : 35819.65
Average P&L per Trade         : 28.12
Average Bars Held             : 62.76
Average Bars Held (Profits)   : 112.87
Average Bars Held (Losses)    : 26.48
Overall ROI                   : 596.99%
Average Profit %              : 2.55%
Average Loss %                : -1.04%
Payoff Ratio (P/L%)           : 2.45
Risk-Reward Ratio (L/P%)      : 1:2.45
Avg Profit/Loss %             : 0.47%
Winners %                     : 41.99%
Losers %                      : 58.01%
Total Profit                  : 81946.60
Total Loss                    : -46126.95
Average Profit                : 153.17
Average Loss                  : -62.42
Largest Win                   : 1254.20
Largest Win Percentage        : 20.90%
Largest Loss                  : -397.75
Bars in Largest Win Trade     : 283
Bars in Largest Loss Trade    : 3
Max Drawdown                  : -1489.35
Max Drawdown %                : -33.810558

In [22]:
kpi_df[kpi_df['Drawdown']==kpi_df['Drawdown'].min()]

,EDT,Exit_Datetime,CE_Strike_Price,PE_Strike_Price,Expiry_Date,EV_CE,EV_PE,EX_CE,EX_PE,CE_PL,PE_PL,Total_PL,Position,No. of Bars Held,ROI,Drawdown,DD_ROI,Cumulative_Max,Drawdown_Percentage
1259,2025-06-23 09:30:00,2025-06-23 11:30:00,24800,24900,2025-06-26,190.3,0.0,246.45,0.0,-56.15,0.0,-56.15,Short,9,-0.009358,-1489.35,-0.248225,37141.15,-4.009973


In [23]:
import pandas as pd
df=o.copy()
# Assuming your dataframe is named `df`
df['Exit_Datetime'] = pd.to_datetime(df['Exit_Datetime'])

# Filter for year 2025
df_2025 = df[df['Exit_Datetime'].dt.year == 2025]
df_2025 = df_2025[df_2025['Position'] == 'Short']

# Create a new column for 'Year-Month'
df_2025['Month'] = df_2025['Exit_Datetime'].dt.to_period('M')

# Group by Month and sum Total_PL
monthly_pl = df_2025.groupby('Month')['CE_PL'].sum().reset_index()


monthly_pl


,Month,CE_PL
0,2025-01,325.25
1,2025-02,460.30
2,2025-03,111.10
3,2025-04,317.60
4,2025-05,-444.25
5,2025-06,-3.05
6,2025-07,369.40


In [24]:
monthly_pl['CE_PL'].sum()

np.float64(1136.35)

In [25]:
new_df=kpi_df[0:1250]
new_df[new_df['Drawdown']==new_df['Drawdown'].min()]

,EDT,Exit_Datetime,CE_Strike_Price,PE_Strike_Price,Expiry_Date,EV_CE,EV_PE,EX_CE,EX_PE,CE_PL,PE_PL,Total_PL,Position,No. of Bars Held,ROI,Drawdown,DD_ROI,Cumulative_Max,Drawdown_Percentage
1249,2025-05-27 11:15:00,2025-05-27 12:45:00,24900,25000,2025-05-29,309.45,199.95,198.25,277.8,-111.2,-77.85,-189.05,Long,7,-0.031508,-1435.65,-0.239275,37141.15,-3.865389


In [26]:
# Ensure EDT is datetime
sample=o.copy()
sample['EDT'] = pd.to_datetime(sample['EDT'])

# Extract year and create a new column
sample['Year'] = sample['EDT'].dt.year

# Group by year and sum PnL
pnl_by_year = sample.groupby('Year')['Total_PL'].sum().reset_index()


pnl_by_year

,Year,Total_PL
0,2012,1587.95
1,2013,2273.40
2,2014,2167.95
3,2015,1615.35
4,2016,1008.40
5,2017,1466.50
6,2018,2097.05
7,2019,1521.00
8,2020,5615.65
9,2021,4451.40


In [27]:
o.to_excel('Self_Made_Strategy.xlsx', sheet_name="Tradesheet", index=False)